# ocr-labins-index.ipynb

This file runs OCR on every page of the LABINS volume index PDFs and stores one `.json` and one `.txt` per volume in `data/labins-ocr/`. Run on a GPU runtime in Google Colab (preferably T4 GPU or higher).

The OCR cell checkpoints by page number, so a rerun resumes where it stopped. The tagging cell then labels each page as `index`, `unsure` or `diagram` from the page-to-page pattern of line confidence: the index span is the opening run of high-confidence pages (meant to capture print index pages), the first page after it is `unsure` (meant to capture the handwritten index pages or the first diagram page), and the rest are `diagram`. The last cell charts the mean line confidence and the percent of lines scoring at or above 90% by page, with the span marked, and shows the last index page next to the unsure page for each volume.

It runs on every volume PDF in `data/labins-index/`.

In [1]:
# setup
CUDA_TAG = "cu126"
import glob, shutil, sysconfig, importlib.util
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# set dir
DATA_DIR = "/content/drive/MyDrive/armed-occupation-act/data"
WHEELS = "/content/drive/MyDrive/armed-occupation-act/wheels"

# cache paddle wheel
if not glob.glob(f"{WHEELS}/paddlepaddle_gpu*.whl"):
  print("downloading paddle wheel")
  !pip download -q --no-deps paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/{CUDA_TAG}/ -d {WHEELS}

# install
if importlib.util.find_spec("paddle") is None:
  print("installing paddle")
  !pip -q install --find-links {WHEELS} paddlepaddle-gpu
  !pip -q install paddleocr pymupdf

# remove torch
if importlib.util.find_spec("torch") is not None:
  print("removing torch")
  !pip -q uninstall -y torch torchvision torchaudio
  site = Path(sysconfig.get_paths()["purelib"])
  for pat in ["torch", "torch-*", "torchgen", "functorch"]:   # pip leaves stubs behind
        for p in site.glob(pat):
            shutil.rmtree(p, ignore_errors=True)
print("torch:", importlib.util.find_spec("torch"))   # should be None

Mounted at /content/drive
installing paddle
removing torch
torch: None


In [2]:
# verify CUDA
!nvidia-smi
import paddle
print("compiled with CUDA:", paddle.device.is_compiled_with_cuda())
print("device:", paddle.device.get_device())
paddle.utils.run_check()
assert paddle.device.is_compiled_with_cuda() and "gpu" in paddle.device.get_device(), \
    "Paddle is not using the GPU!"

Wed Nov 11 19:27:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


compiled with CUDA: True
device: gpu:0
Running verify PaddlePaddle program ... 


/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


In [6]:
# imports
import os
os.environ.update({"GLOG_minloglevel": "3", "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True"})
import json, logging, warnings, threading
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pymupdf
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from paddleocr import PaddleOCR

# set dir
DATA_DIR = "/content/drive/MyDrive/armed-occupation-act/data"
IN_DIR = Path(DATA_DIR) / "labins-index"
OUT_DIR = Path(DATA_DIR) / "labins-ocr"
OUT_DIR.mkdir(exist_ok=True)

# config
VOLS = sorted(int(p.stem[3:]) for p in IN_DIR.glob("vol*.pdf"))   # every pdf in the folder
LONG_SIDE = 4000     # render size in px, set from page size so dpi is not assumed
BATCH = 20           # pages per gpu call
RENDER_WORKERS = 4   # cpu threads rendering the next batch

# helpers
def save(path, text):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)                                    # atomic write

def render(doc, i):
    zoom = LONG_SIDE / max(doc[i].rect.width, doc[i].rect.height)
    pix = doc[i].get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
    return np.array(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))

_tl = threading.local()
def render_page(name, i):
    cur = getattr(_tl, "cur", None)
    if cur is None or cur[0] != name:                    # one open pdf per thread, swapped when the volume changes
        if cur:
            cur[1].close()
        _tl.cur = cur = (name, pymupdf.open(IN_DIR / f"{name}.pdf"))
    return render(cur[1], i)

def to_record(i, img, r):
    boxes = r["rec_boxes"] if "rec_boxes" in r else [[*np.min(p, 0).tolist(), *np.max(p, 0).tolist()] for p in r["rec_polys"]]
    h, w = img.shape[:2]
    lines = [{"text": t, "score": round(float(s), 4), "box": np.asarray(b).tolist()}   # box is x0,y0,x1,y1 in pixels
             for t, s, b in zip(r["rec_texts"], r["rec_scores"], boxes)]
    return {"page": i + 1, "width": w, "height": h, "lines": lines}

def write_vol(name, recs):
    recs = sorted(recs, key=lambda r: r["page"])
    save(OUT_DIR / f"{name}.json", json.dumps(recs, ensure_ascii=False))
    save(OUT_DIR / f"{name}.txt", "".join(
        f"\n\n===== PAGE {r['page']} =====\n" + "\n".join(l["text"] for l in r["lines"]) for r in recs))

# model
ocr = PaddleOCR(device="gpu",
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False)

# plan batches
recs_by, jobs = {}, []
for v in VOLS:
    name = f"vol{v:03d}"
    with pymupdf.open(IN_DIR / f"{name}.pdf") as d:
        n_pages = len(d)
    jf = OUT_DIR / f"{name}.json"
    recs = {r["page"]: r for r in json.loads(jf.read_text(encoding="utf-8"))} if jf.exists() else {}   # resume by page number
    todo = [i for i in range(n_pages) if i + 1 not in recs]
    print(f"{name}: {n_pages} pages, {n_pages - len(todo)} already done, {len(todo)} to go")
    recs_by[name] = recs
    jobs += [(name, todo[k:k + BATCH]) for k in range(0, len(todo), BATCH)]   # batches never span volumes

# ocr w cpu overlap
with ThreadPoolExecutor(RENDER_WORKERS) as pool, tqdm(total=sum(len(idx) for _, idx in jobs), unit="page") as bar:
    submit = lambda job: [pool.submit(render_page, job[0], i) for i in job[1]]
    pending = submit(jobs[0]) if jobs else []
    for n, (name, idx) in enumerate(jobs):
        imgs = [f.result() for f in pending]
        pending = submit(jobs[n + 1]) if n + 1 < len(jobs) else []   # render next batch while gpu runs
        recs = recs_by[name]
        for i, img, r in zip(idx, imgs, ocr.predict(imgs)):
            recs[i + 1] = to_record(i, img, r)
        write_vol(name, list(recs.values()))             # checkpoint per batch
        bar.set_description(name)
        bar.update(len(idx))

vol001: 8 pages, 8 already done, 0 to go
vol002: 19 pages, 19 already done, 0 to go
vol003: 22 pages, 22 already done, 0 to go
vol004: 38 pages, 38 already done, 0 to go
vol005: 11 pages, 11 already done, 0 to go
vol006: 19 pages, 19 already done, 0 to go
vol007: 8 pages, 8 already done, 0 to go
vol008: 16 pages, 16 already done, 0 to go
vol009: 14 pages, 14 already done, 0 to go
vol010: 12 pages, 12 already done, 0 to go
vol011: 10 pages, 10 already done, 0 to go
vol012: 13 pages, 13 already done, 0 to go
vol013: 11 pages, 11 already done, 0 to go
vol014: 7 pages, 7 already done, 0 to go
vol015: 12 pages, 12 already done, 0 to go
vol016: 26 pages, 26 already done, 0 to go
vol017: 8 pages, 8 already done, 0 to go
vol018: 8 pages, 8 already done, 0 to go
vol019: 22 pages, 22 already done, 0 to go
vol020: 8 pages, 8 already done, 0 to go
vol021: 14 pages, 14 already done, 0 to go
vol022: 9 pages, 9 already done, 0 to go
vol023: 9 pages, 9 already done, 0 to go
vol024: 21 pages, 21 alread

  0%|          | 0/643 [00:00<?, ?page/s]

In [7]:
# imports
import json
import numpy as np
from pathlib import Path

# set dir
DATA_DIR = "/content/drive/MyDrive/armed-occupation-act/data"
IN_DIR = Path(DATA_DIR) / "labins-index"
OUT_DIR = Path(DATA_DIR) / "labins-ocr"

# config
VOLS = sorted(int(p.stem[3:]) for p in IN_DIR.glob("vol*.pdf"))   # every pdf in the folder
LINE_MIN = 0.90      # a line counts as confident at or above this score
MEAN_HIGH = 0.98     # page is high if mean line score is at least this
SHARE_HIGH = 0.95    # or if share of confident lines is at least this
MAX_GAP = 1          # non-high pages allowed inside the opening run
MAX_LEAD = 2         # non-high pages before the first high page that still count as index

# helpers
def measures(rec):
    s = np.array([l["score"] for l in rec["lines"]], dtype=float)
    if len(s) == 0:
        return 0, 0.0, 0.0                               # empty page scores 0
    return len(s), float(s.mean()), float((s >= LINE_MIN).mean())

def find_span(high):
    if True not in high:
        return None
    first = high.index(True)
    if first > MAX_LEAD:
        return None                                      # index not at the front
    last, gap = first, 0
    for j in range(first + 1, len(high)):
        if high[j]:
            last, gap = j, 0                             # dip inside the run was noise
        else:
            gap += 1
            if gap > MAX_GAP:
                break
    return last + 1                                      # last index page, 1-based

# tag pages
flagged = []
for v in VOLS:
    name = f"vol{v:03d}"
    recs = sorted(json.loads((OUT_DIR / f"{name}.json").read_text(encoding="utf-8")), key=lambda r: r["page"])
    for r in recs:
        r["n_lines"], mean, share = measures(r)
        r["mean"], r["share"] = round(mean, 4), round(share, 4)
    k = find_span([r["mean"] >= MEAN_HIGH or r["share"] >= SHARE_HIGH for r in recs])   # either metric, for recall
    for r in recs:
        if k is None:
            r["tag"] = "unsure"
        else:
            r["tag"] = "index" if r["page"] <= k else "unsure" if r["page"] == k + 1 else "diagram"
    tmp = (OUT_DIR / f"{name}.tmp")
    tmp.write_text(json.dumps(recs, ensure_ascii=False), encoding="utf-8")
    tmp.replace(OUT_DIR / f"{name}.json")
    if k is None:
        flagged.append(v)
        print(f"{name}: {len(recs)} pages, no index span found, all unsure")
    else:
        print(f"{name}: {len(recs)} pages, index 1-{k}, unsure {k + 1 if k < len(recs) else '-'}, diagram {k + 2}-{len(recs)}")
print("flagged:", flagged)

vol001: 8 pages, index 1-1, unsure 2, diagram 3-8
vol002: 19 pages, index 1-3, unsure 4, diagram 5-19
vol003: 22 pages, index 1-3, unsure 4, diagram 5-22
vol004: 38 pages, index 1-8, unsure 9, diagram 10-38
vol005: 11 pages, index 1-2, unsure 3, diagram 4-11
vol006: 19 pages, index 1-3, unsure 4, diagram 5-19
vol007: 8 pages, index 1-2, unsure 3, diagram 4-8
vol008: 16 pages, index 1-3, unsure 4, diagram 5-16
vol009: 14 pages, index 1-3, unsure 4, diagram 5-14
vol010: 12 pages, index 1-3, unsure 4, diagram 5-12
vol011: 10 pages, index 1-3, unsure 4, diagram 5-10
vol012: 13 pages, index 1-4, unsure 5, diagram 6-13
vol013: 11 pages, index 1-2, unsure 3, diagram 4-11
vol014: 7 pages, index 1-2, unsure 3, diagram 4-7
vol015: 12 pages, index 1-3, unsure 4, diagram 5-12
vol016: 26 pages, index 1-4, unsure 5, diagram 6-26
vol017: 8 pages, index 1-4, unsure 5, diagram 6-8
vol018: 8 pages, index 1-2, unsure 3, diagram 4-8
vol019: 22 pages, index 1-4, unsure 5, diagram 6-22
vol020: 8 pages, inde

In [9]:
# setup
import json
import numpy as np
import pymupdf
import matplotlib.pyplot as plt
from pathlib import Path

# set dir
DATA_DIR = "/content/drive/MyDrive/armed-occupation-act/data"
IN_DIR = Path(DATA_DIR) / "labins-index"
OUT_DIR = Path(DATA_DIR) / "labins-ocr"
CHECK_DIR = Path(DATA_DIR) / "labins-ocr-check"
CHECK_DIR.mkdir(exist_ok=True)

# config
VOLS = sorted(int(p.stem[3:]) for p in IN_DIR.glob("vol*.pdf"))   # every pdf in the folder
LINE_MIN = 0.90      # a line counts as confident at or above this score
SHOW_PX = 1100       # long side of saved page in px

# page image
def page_img(doc, i):
    zoom = SHOW_PX / max(doc[i].rect.width, doc[i].rect.height)
    pix = doc[i].get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
    return np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, 3)

# chart and boundary pages
for v in VOLS:
    recs = sorted(json.loads((OUT_DIR / f"vol{v:03d}.json").read_text(encoding="utf-8")), key=lambda r: r["page"])
    doc = pymupdf.open(IN_DIR / f"vol{v:03d}.pdf")
    x = [r["page"] for r in recs]
    mean = np.array([r["mean"] for r in recs])
    share = np.array([r["share"] for r in recs])
    idx = [r["page"] for r in recs if r["tag"] == "index"]
    uns = [r["page"] for r in recs if r["tag"] == "unsure"]

    # line chart with span marked
    fig, ax = plt.subplots(figsize=(10, 4))
    if idx:
        ax.axvspan(min(idx) - 0.5, max(idx) + 0.5, color="tab:green", alpha=0.15, label="index span")
    for p in uns:
        ax.axvspan(p - 0.5, p + 0.5, color="tab:red", alpha=0.2, label="unsure")
    ax.plot(x, mean * 100, marker="o", label="mean line confidence")
    ax.plot(x, share * 100, marker="s", label=f"lines at or above {LINE_MIN:.0%}")
    ax.set(xlabel="page", ylabel="percent", title=f"vol {v}: {len(recs)} pages", ylim=(0, 102))
    h, l = ax.get_legend_handles_labels()
    ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys())   # one entry per label
    ax.grid(alpha=0.3)
    fig.savefig(CHECK_DIR / f"vol{v:03d}_chart.png", dpi=100, bbox_inches="tight")
    plt.close(fig)                                                # free memory across volumes

    # last index page next to unsure page
    if idx and uns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 9))
        for ax, p in zip(axes, (max(idx), uns[0])):
            ax.axis("off")
            ax.imshow(page_img(doc, p - 1))
            r = recs[p - 1]
            ax.set_title(f"vol {v} page {p} ({r['tag']}): mean {r['mean']*100:.1f}%, lines >= {LINE_MIN:.0%}: {r['share']*100:.1f}%")
        fig.tight_layout()
        fig.savefig(CHECK_DIR / f"vol{v:03d}_pair.png", dpi=100)
        plt.close(fig)
    doc.close()